# 🌊 Urban Flood Risk Prediction: Global City Analysis 2025

## 1. Environment Setup

In [ ]:
# Core Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

# Machine Learning Libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.metrics import (
    r2_score, mean_squared_error, mean_absolute_error,
    accuracy_score
)

# Configuration
warnings.filterwarnings('ignore')
np.random.seed(42)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

# Plot Styling
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Environment setup complete")

## 2. Data Loading

In [ ]:
# Auto-discover dataset files
print("🔍 Scanning input directory...\n")
all_csv_files = []

for dirname, _, filenames in os.walk('./data'):
    for filename in filenames:
        if filename.endswith('.csv'):
            filepath = os.path.join(dirname, filename)
            all_csv_files.append(filepath)
            print(f"   Found: {filepath}")

if len(all_csv_files) == 0:
    raise FileNotFoundError("❌ No CSV files found!")

# Load dataset
csv_path = all_csv_files[0]
print(f"\n✅ Loading: {csv_path}")
df = pd.read_csv(csv_path, low_memory=False)
print(f"✅ Shape: {df.shape}")

## 3. Quick Data Overview

In [ ]:
print("="*80)
print("📊 DATA OVERVIEW")
print("="*80)
print(f"\nRows: {df.shape[0]:,} | Columns: {df.shape[1]}")
print(f"\nColumn Names:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

print(f"\nMissing Values: {df.isnull().sum().sum():,}")
print(f"Duplicates: {df.duplicated().sum():,}")
print(f"\nFirst 3 rows:")
display(df.head(3))

## 4. Fast Data Cleaning

In [ ]:
print("\n🧹 FAST DATA CLEANING")
print("="*80)

df_clean = df.copy()

# Step 1: Drop columns with >50% missing
missing_pct = df_clean.isnull().sum() / len(df_clean)
high_missing_cols = missing_pct[missing_pct > 0.5].index.tolist()
if len(high_missing_cols) > 0:
    df_clean = df_clean.drop(columns=high_missing_cols)
    print(f"✅ Dropped {len(high_missing_cols)} high-missing columns")

# Step 2: Fill numerical columns with median
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df_clean[col].isnull().any():
        df_clean[col]= df_clean[col].fillna(df_clean[col].median())
print(f"✅ Filled {len(numeric_cols)} numerical columns")

# Step 3: Fill categorical columns with mode
cat_cols = df_clean.select_dtypes(include=['object', 'category']).columns
for col in cat_cols:
    if df_clean[col].isnull().any():
        mode_val = df_clean[col].mode()
        fill_val = mode_val[0] if len(mode_val) > 0 else 'Unknown'
        df_clean[col] = df_clean[col].fillna(fill_val)
print(f"✅ Filled {len(cat_cols)} categorical columns")

# Step 4: Drop remaining rows with missing
rows_before = len(df_clean)
df_clean = df_clean.dropna()
print(f"✅ Dropped {rows_before - len(df_clean)} rows with remaining NaN")

# Step 5: Drop duplicates
df_clean = df_clean.drop_duplicates()

# Final verification
final_missing = df_clean.isnull().sum().sum()
assert final_missing == 0, f"❌ {final_missing} missing values remain!"

print(f"\n✅ CLEANING COMPLETE")
print(f"Final shape: {df_clean.shape}")
print(f"Data retained: {100*len(df_clean)/len(df):.1f}%")

## 5. Target Identification & EDA

In [ ]:
# Identify target column
target_col = None
for col in df_clean.columns:
    if any(kw in col.lower() for kw in ['risk', 'score', 'level', 'target', 'label']):
        target_col = col
        break

if target_col is None:
    target_col = df_clean.columns[-1]

print(f"\n🎯 Target Column: '{target_col}'")
print("="*80)

# Quick target analysis
if df_clean[target_col].dtype in [np.float64, np.float32, np.int64, np.int32] and df_clean[target_col].nunique() > 20:
    print(f"Type: REGRESSION (continuous)")
    print(f"Range: [{df_clean[target_col].min():.2f}, {df_clean[target_col].max():.2f}]")
    print(f"Mean: {df_clean[target_col].mean():.2f}")
else:
    print(f"Type: CLASSIFICATION (categorical)")
    print(f"Classes: {df_clean[target_col].nunique()}")
    print(f"\nTop 5 classes:")
    print(df_clean[target_col].value_counts().head())

## 6. Fast Feature Engineering

In [ ]:
print("\n🔧 FEATURE ENGINEERING")
print("="*80)

df_feat = df_clean.copy()

# Encode ALL categorical columns (including target)
cat_cols = df_feat.select_dtypes(include=['object', 'category']).columns.tolist()

if len(cat_cols) > 0:
    le = LabelEncoder()
    for col in cat_cols:
        df_feat[col] = le.fit_transform(df_feat[col].astype(str))
    print(f"✅ Encoded {len(cat_cols)} categorical columns")

# Verify all numeric
assert df_feat.select_dtypes(exclude=[np.number]).shape[1] == 0, "Non-numeric columns remain!"
print(f"✅ All columns now numeric")

# Prepare X and y
feature_cols = [col for col in df_feat.columns if col != target_col]
X = df_feat[feature_cols].copy()
y = df_feat[target_col].copy()

print(f"\n✅ Features: {X.shape[1]} | Samples: {len(X)}")

## 7. Pre-Modeling Checks

In [ ]:
print("\n🔍 PRE-MODELING CHECKS")
print("="*80)

# Check 1: Missing values
x_missing = X.isnull().sum().sum()
y_missing = y.isnull().sum()
print(f"X missing: {x_missing} | y missing: {y_missing}")
assert x_missing == 0 and y_missing == 0, "Missing values detected!"

# Check 2: Infinite values
inf_count = np.isinf(X.select_dtypes(include=[np.number])).sum().sum()
print(f"Infinite values: {inf_count}")
if inf_count > 0:
    X = X.replace([np.inf, -np.inf], np.nan)
    for col in X.columns:
        X[col].fillna(X[col].median(), inplace=True)
    print("✅ Fixed infinite values")

# Check 3: Problem type
is_regression_problem = y.nunique() > 20
problem_type = "REGRESSION" if is_regression_problem else "CLASSIFICATION"
print(f"\n✅ Problem Type: {problem_type}")
print(f"✅ All checks passed - Ready for modeling")

## 8. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"✅ Train: {len(X_train)} samples | Test: {len(X_test)} samples")

## 9. Fast Model Training (2 Models Only)

In [ ]:
# Simple evaluation function
def evaluate_fast(model, X_tr, X_te, y_tr, y_te, name, is_reg):
    print(f"\n{'='*70}")
    print(f"🔍 {name}")
    print(f"{'='*70}")
    
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    
    if is_reg:
        r2 = r2_score(y_te, y_pred)
        rmse = np.sqrt(mean_squared_error(y_te, y_pred))
        mae = mean_absolute_error(y_te, y_pred)
        print(f"R² Score: {r2:.4f}")
        print(f"RMSE: {rmse:.4f}")
        print(f"MAE: {mae:.4f}")
        return {'name': name, 'model': model, 'r2': r2, 'rmse': rmse, 'mae': mae}
    else:
        acc = accuracy_score(y_te, y_pred)
        print(f"Accuracy: {acc:.4f}")
        return {'name': name, 'model': model, 'accuracy': acc}

print("✅ Evaluation function ready")

In [ ]:
# MODEL 1: Random Forest (Fast & Reliable)
if is_regression_problem:
    model1 = RandomForestRegressor(
        n_estimators=50, max_depth=10, min_samples_split=10,
        random_state=42, n_jobs=-1
    )
else:
    model1 = RandomForestClassifier(
        n_estimators=50, max_depth=10, min_samples_split=10,
        random_state=42, n_jobs=-1
    )

result1 = evaluate_fast(model1, X_train, X_test, y_train, y_test, "Random Forest", is_regression_problem)

In [ ]:
# MODEL 2: Gradient Boosting (Optimized)
n_samples = len(X_train)
n_est = 30 if n_samples > 10000 else 50
max_d = 3 if n_samples > 10000 else 5

if is_regression_problem:
    model2 = GradientBoostingRegressor(
        n_estimators=n_est, learning_rate=0.1, max_depth=max_d,
        subsample=0.8, random_state=42
    )
else:
    model2 = GradientBoostingClassifier(
        n_estimators=n_est, learning_rate=0.1, max_depth=max_d,
        subsample=0.8, random_state=42
    )

result2 = evaluate_fast(model2, X_train, X_test, y_train, y_test, "Gradient Boosting", is_regression_problem)

## 10. Model Comparison

In [ ]:
print("\n" + "="*80)
print("📊 MODEL COMPARISON")
print("="*80 + "\n")

results = [result1, result2]
comparison_df = pd.DataFrame(results)

if is_regression_problem:
    comparison_df = comparison_df.sort_values('r2', ascending=False)
    display(comparison_df[['name', 'r2', 'rmse', 'mae']])
    best_model = comparison_df.iloc[0]['name']
    print(f"\n🏆 BEST MODEL: {best_model}")
    print(f"   R² Score: {comparison_df.iloc[0]['r2']:.4f}")
else:
    comparison_df = comparison_df.sort_values('accuracy', ascending=False)
    display(comparison_df[['name', 'accuracy']])
    best_model = comparison_df.iloc[0]['name']
    print(f"\n🏆 BEST MODEL: {best_model}")
    print(f"   Accuracy: {comparison_df.iloc[0]['accuracy']:.4f}")

## 11. Feature Importance

In [ ]:
# Get best model
best_result = comparison_df.iloc[0]
best_model_obj = best_result['model']

if hasattr(best_model_obj, 'feature_importances_'):
    print("\n🔍 FEATURE IMPORTANCE")
    print("="*80 + "\n")
    
    importances = best_model_obj.feature_importances_
    imp_df = pd.DataFrame({
        'Feature': feature_cols,
        'Importance': importances
    }).sort_values('Importance', ascending=False)
    
    print("Top 15 Features:\n")
    display(imp_df.head(15))
    
    # Plot
    plt.figure(figsize=(10, 6))
    top_n = min(15, len(imp_df))
    top_feat = imp_df.head(top_n)
    plt.barh(range(top_n), top_feat['Importance'], color='steelblue')
    plt.yticks(range(top_n), top_feat['Feature'])
    plt.xlabel('Importance')
    plt.title(f'Top {top_n} Features - {best_model}')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("\n⚠️ Feature importance not available for this model")